# City C (Montreal): Initial Model Evaluation

**Objective.** Evaluate the original Toronto/Vancouver price model on Montreal with **no retraining and no changes to preprocessing or model features**.

The original ETL was run with `city = montreal`. The silver-layer run reported **9,058** Montreal rows. This notebook does **not** write to or modify existing lakehouse tables. If a step fails, we preserve the error/output before implementing any fixes.

**Frozen model:** MLflow model logged by `03_price_model` (run `RandomForest_Price_Model`). The model predicts **log(1 + price)**, so convert with `np.expm1` before calculating MAE in CAD.

## 1. Setup and original A/B reference

In [0]:
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.metrics import mean_absolute_error
from pyspark.sql import functions as F

CITY_C = "montreal"
GOLD_TABLE = "airbnb_lakehouse.gold_listings_features"
MODEL_ID = "m-f6812ff886404598946420b8d3ea713a"
MODEL_URI = f"models:/{MODEL_ID}"

# Reported held-out A/B results from the original 03_price_model notebook.
# Reference values only; NOT recalculated from the full predictions table.
AB_REFERENCE_MAE = {"toronto": 61.24, "vancouver": 92.25, "combined": 68.53}
print("Frozen model URI:", MODEL_URI)
print("Original A/B held-out MAEs (CAD):", AB_REFERENCE_MAE)

## 2. Load Montreal gold features

We use the existing gold feature table as-is.

In [0]:
df_c_spark = spark.table(GOLD_TABLE).filter(F.col("city") == CITY_C)
print("Montreal gold rows:", df_c_spark.count())
assert df_c_spark.count() > 0, "No Montreal rows in gold; investigate ETL before scoring."
df_c = df_c_spark.toPandas()
print("Data shape:", df_c.shape)
display(df_c.head(5))

## 3. Inspect model feature compatibility

These are the **exact 13 model inputs** from `03_price_model.ipynb`. Diagnostics are read-only.

In [0]:
cat_features = [
    "room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost"
]
num_features = [
    "accommodates", "bedrooms", "beds", "bathrooms_clean",
    "latitude", "longitude", "hosts_time_as_host_years",
    "minimum_nights", "availability_365"
]
feature_cols = num_features + cat_features
required_cols = feature_cols + ["price_clean"]
missing_columns = sorted(set(required_cols) - set(df_c.columns))
print("Missing required columns:", missing_columns)
assert not missing_columns, f"Schema mismatch: {missing_columns}"

checks = pd.DataFrame({
    "feature": required_cols,
    "dtype": [str(df_c[c].dtype) for c in required_cols],
    "missing_count": [int(df_c[c].isna().sum()) for c in required_cols],
    "missing_pct": [round(100 * df_c[c].isna().mean(), 2) for c in required_cols]
})
display(checks)
print("Rows with at least one missing feature:", int(df_c[feature_cols].isna().any(axis=1).sum()))

## 4. Load the original fitted model

The MLflow model should be an `sklearn.pipeline.Pipeline` with `preprocessor` and `regressor` steps.

In [0]:
model_pipeline = mlflow.sklearn.load_model(MODEL_URI)
print("Loaded model type:", type(model_pipeline))
print("Pipeline steps:", list(model_pipeline.named_steps))
assert {"preprocessor", "regressor"}.issubset(model_pipeline.named_steps), "Unexpected fitted pipeline structure"
print("Random Forest:", model_pipeline.named_steps["regressor"])
print("Model loaded; no retraining performed.")

## 5. First unchanged prediction on Montreal

**Checkpoint:** This is the first actual model transfer test.

In [0]:
# The source model was fitted on log1p(price_clean).
# Use exactly the original feature columns and the original fitted preprocessing.
X_c = df_c[feature_cols]
y_c = df_c["price_clean"].to_numpy(dtype=float)

pred_log_c = model_pipeline.predict(X_c)
pred_price_c = np.expm1(pred_log_c)

print("Predictions generated:", len(pred_price_c), "/", len(df_c))
assert len(pred_price_c) == len(df_c)
assert np.isfinite(pred_price_c).all(), "Non-finite predictions; record before any corrections"
print("Initial unchanged scoring completed successfully.")

## 6. Compute initial Montreal MAE (CAD)

In [0]:
assert np.isfinite(y_c).all(), "Non-finite true prices; investigate before calculating MAE"
mae_c_original = mean_absolute_error(y_c, pred_price_c)
print(f"Original A/B model on Montreal: MAE = {mae_c_original:.2f} CAD")

comparison = pd.DataFrame([
    {"evaluation": "Original model: Toronto held-out", "MAE_CAD": AB_REFERENCE_MAE["toronto"], "source": "03_price_model (reported)"},
    {"evaluation": "Original model: Vancouver held-out", "MAE_CAD": AB_REFERENCE_MAE["vancouver"], "source": "03_price_model (reported)"},
    {"evaluation": "Original model: A/B combined held-out", "MAE_CAD": AB_REFERENCE_MAE["combined"], "source": "03_price_model (reported)"},
    {"evaluation": "Original model: Montreal transfer", "MAE_CAD": round(mae_c_original, 2), "source": "This notebook, all Montreal gold rows"},
])
display(comparison)

# Local output only; do NOT overwrite gold_model_predictions.
results_c = df_c[["id", "city", "room_type", "accommodates", "price_clean"]].copy()
results_c["predicted_price"] = pred_price_c
results_c["absolute_error"] = np.abs(y_c - pred_price_c)
display(results_c.head(10))

# 7. Table consistency analysis

In [0]:
%sql
SELECT 'Features' AS source, city, COUNT(*) AS listings
FROM airbnb_lakehouse.gold_listings_features
GROUP BY city

UNION ALL

SELECT 'Predictions', city, COUNT(*)
FROM airbnb_lakehouse.gold_model_predictions
GROUP BY city

UNION ALL

SELECT 'Genie analytics', city, COUNT(*)
FROM airbnb_lakehouse.gold_listings_analytics
GROUP BY city

ORDER BY source, city;

## 8. Initial Results and Analysis

### 8.1. ETL and Feature Compatibility

The existing city-parameterized ETL pipeline successfully processed Montreal without code modifications, producing **9,058 listings** in the gold feature table.

All 13 features required by the trained model were present, with no missing values detected. The existing preprocessing pipeline accepted Montreal's data without additional cleaning or feature transformations.

The original Toronto and Vancouver listing counts remained unchanged after onboarding Montreal.

**Result:** No ETL or feature compatibility fixes were required.

### 8.2. Original Model Evaluation

The original Random Forest model, trained exclusively on Toronto and Vancouver, was loaded from MLflow and applied to Montreal without retraining or modification.

The model successfully generated predictions for all **9,058 Montreal listings**, achieving an MAE of **79.68 CAD**.

| Evaluation dataset | MAE (CAD) |
|---|---:|
| Toronto (held-out) | 61.24 |
| Vancouver (held-out) | 92.25 |
| Toronto + Vancouver (held-out) | 68.53 |
| Montreal (unseen city) | **79.68** |

Montreal's MAE is approximately **16.3% higher** than the combined Toronto/Vancouver held-out MAE, but lower than Vancouver's individual MAE.

This indicates that the original model can transfer to an unseen city without major technical difficulties. However, the comparison is not fully equivalent: the A/B results were obtained from held-out samples of cities represented during training, while Montreal is an entirely unseen geographic domain.

The predictive quality of the transferred model still needs to be assessed against the original median-based baseline.

### 8.3. Dashboard and Genie Integration

The initial integration tests revealed different levels of compatibility across the downstream components.

**Dashboard.** General listing statistics and aggregations successfully incorporated Montreal. However, Montreal predictions were absent from model-performance visualizations.

**Genie Agent.** The agent incorrectly reported that Montreal had zero listings and excluded Montreal from city-level price comparisons, despite the successful ETL execution.

The SQL investigation in Section 7 confirmed the underlying inconsistency:

| Data source | Toronto | Vancouver | Montreal |
|---|---:|---:|---:|
| `gold_listings_features` | 16,725 | 5,145 | 9,058 |
| `gold_model_predictions` | 16,725 | 5,145 | — |
| `gold_listings_analytics` | 16,725 | 5,145 | — |

The curated `gold_listings_analytics` view used by Genie is based on `gold_model_predictions`, joined with `gold_listings_features`. Since Montreal predictions have not been published, Montreal is also absent from this view.

**Result:** The downstream integration is incomplete, despite successful ETL processing and model inference.

### 8.4. Architectural and Design Limitations

The investigation identified an additional limitation in the existing implementation.

The original `03_price_model.ipynb` combines several logically distinct responsibilities:

1. Training-data preparation and baseline construction.
2. Model definition and training.
3. Model evaluation.
4. Full-dataset inference and prediction publication.

This design creates unnecessary dependencies between training and inference. Although the trained model is available through MLflow, generating and publishing predictions for a new city requires reconstructing intermediate calculations from the training notebook or partially reusing its code.

Moreover, the existing publication step overwrites the entire `gold_model_predictions` table rather than supporting city-specific updates. Rerunning it for Montreal would therefore risk replacing existing Toronto/Vancouver results and could unnecessarily repeat model training.

The absence of a standalone inference and publication workflow is a **design and extensibility limitation**, rather than a failure of the trained model itself.

A better separation would allow new cities to be scored using the saved model and published independently of the training process.

### 8.5. Summary of Initial Findings

| Component | Initial result | Required action |
|---|---|---|
| ETL pipeline | Successful | None |
| Feature compatibility | Successful | None |
| Original model inference | Successful; MAE = 79.68 CAD | Evaluate against baseline |
| Dashboard — general statistics | Successful | None |
| Dashboard — model predictions | Montreal absent | Publish Montreal predictions |
| Genie Agent | Incorrectly excludes Montreal | Restore data coverage and retest |
| Prediction publication workflow | Coupled to training notebook | Introduce independent city-specific scoring/publication |

### 8.6. Planned Improvements

The following changes will be implemented separately in `04_2_city_c_fixes.ipynb`, preserving the original evaluation results:

1. Reconstruct the original baseline using Toronto/Vancouver training data and evaluate it on Montreal.
2. Prepare Montreal prediction records using the existing fitted model and preprocessing pipeline.
3. Publish Montreal predictions to `gold_model_predictions` without replacing Toronto/Vancouver records.
4. Verify that Montreal becomes available through `gold_listings_analytics` and retest Genie.
5. Validate Montreal model-performance visualization.
6. Investigate prediction errors and consider model improvements only if justified by the baseline comparison and subsequent analysis.

**Conclusion.** The original ETL pipeline and trained model successfully support Montreal without modifications. The primary issues arise in the downstream integration and the coupling of model training with inference and publication. The next phase will address these limitations while preserving the original model and its evaluation results.